# Tratamento dos Dados 🪢

## Explorando as estatísticas de gênero do Banco Mundial

<pre>Seguiremos investigando as estatísticas relacionadas a gênero na bases de dados do Banco Mundial.</pre>

<pre>Hoje, vamos explorar a série histórica das taxas de fecundidade entre jovens, agrupadas por região geográfica, que foram utilizadas no artigo <a href="https://genderdata.worldbank.org/data-stories/adolescent-fertility/">The Social and Educational Consequences of Adolescent Childbearing</a>.

In [1]:
import wbgapi as wb
import pandas as pd

<pre>Vamos buscar e organizar os códigos e nomes das regiões geográficas da base de dados.</pre>

In [2]:
region_info = wb.region.info()

<pre>A partir do objeto retornado pela consulta, crie um dicionário chamado <b>region_dict</b>.</pre>

<pre>Este dicionário conterá <b>dois</b> itens:
    1 - Chave <b>region_code</b>, cujo objeto associado é uma lista que contém os códigos das regiões geográficas; e 
    2 - Chave <b>region_name</b>, cujo objeto associado é uma lista que contém os nomes das regiões geográficas.
</pre>

👉 dica: explore o atributo `items` do objeto `region_info`.

<details>
  <summary>Resposta</summary>

<br/>

```python
region_dict = {"region_code": [region['code'] for region in region_info.items],
               "region_name": [region['name'] for region in region_info.items]}
```

<br/>

</details>

<pre>Com este formato de dicionário, você consegue facilmente criar um DataFrame.</pre>

<pre>A função <b>Dataframe</b>, do Pandas 🐼, automaticamente interpreta as chaves do dicionário como nomes de colunas, e as listas associadas preencherão as células das colunas correspondentes.</pre>

<pre>Crie o DataFrame <b>region_df</b> a partir do dicionário <b>region_dict</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
region_df = pd.DataFrame(region_dict)
```

<br/>

</details>

<pre>Vamos consultar a série histórica completa <b>SP.ADO.TFRT</b> referente à taxa de fecundidade entre jovens de 15 a 19 anos, armazenando o resultado na variável <b>serie_afr</b>.</pre>

<pre>Não queremos que a consulta retorne dados de países ou de grupos de renda. Portanto, iremos restringir a consulta a regiões geográficas, somente.</pre>

<pre>Para isso, vamos passar, no argumento <b>economy</b>, os códigos das regiões (<b>region_code</b>) que estão presentes no DataFrame <b>region_df</b>, que você acabou de criar.</pre>

In [ ]:
serie_afr = wb.data.DataFrame(series = ['SP.ADO.TFRT'], 
                              economy = region_df['region_code'],
                              time = "all")

<pre>Inspecione a base de dados armazenada na variável <b>serie_afr</b>.</pre>

In [ ]:
serie_afr.head()

<pre>Veja que o formato padrão de retorno dessa consulta é um DataFrame com os códigos de país/região nos índices e os anos nas colunas.</pre>

<pre>Exclua, do DataFrame <b>serie_afr</b>, as colunas em que <b>todos</b> os valores da coluna estão ausentes (no caso, <b>NaN</b>).</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_afr = serie_afr.dropna(axis = 'columns', how = 'all')
```

<br/>

</details>

### Pivoteamento 🥈

#### Pivoteamento do Formato Largo em Formato Longo

<pre>Vamos reorganizar o DataFrame <b>serie_afr</b>, de modo a manter as regiões como índices, os anos em uma coluna <b>year</b> e as taxas em uma coluna <b>af_rate</b>, atribuindo o resultado à variável <b>serie_afr_long</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_afr_long = serie_afr.melt(var_name = "year", value_name = "af_rate", ignore_index = False)
```

<br/>

</details>

### Junção, Mescla e Concatenação 🥇

#### Mescla

##### Inner Join

<pre>Agora, entrará em cena o DataFrame <b>region_df</b>, que você criou há pouco.</pre>

<pre>Adicione ao DataFrame <b>serie_afr_long</b>, de taxas de fecundidade, uma coluna com os respectivos nomes das regiões.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_afr_long = serie_afr_long.merge(right = region_df, 
                                      how = "inner", 
                                      left_on = "economy", 
                                      right_on = "region_code")
```

<br/>

</details>

<pre>Após a junção das tabelas, coloque a coluna com os códigos das regiões (<b>region_code</b>) como índice do DataFrame <b>serie_afr_long</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_afr_long = serie_afr_long.set_index("region_code")
```

<br/>

</details>

<pre>Por fim, formate os valores da coluna <b>year</b>, mantendo somente os números. Ademais, não se esqueça de converter a coluna <b>year</b> em números inteiros.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_afr_long["year"] = serie_afr_long["year"].map(lambda x: x.strip("YR")).astype(int)
```

<br/>

</details>

### Explorando as estatísticas brasileiras de comércio exterior 

<pre>Execute a célula abaixo para criar os 2 (dois) arquivos <b>CSV</b> com as importações, exportações e saldo comercial bilateral do Brasil com seus parceiros comerciais, dos últimos 10 anos.</pre>

In [4]:
import requests
import os
import datetime
import time
import pandas as pd

# desabilitar a verificação de certificado de segurança para fins didáticos
# é possível, considerando que estamos tratando apenas de dados públicos

requests.packages.urllib3.disable_warnings(requests.packages.urllib3.exceptions.InsecureRequestWarning)

# função criada para criar o link, conforme padrão
def cria_url(fluxo, ano):
    url_base = "https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/"
    return f"{url_base}{fluxo.upper()}_{ano}.csv"

def download_file(url):
    response = requests.get(url, verify = False) 
    
    if response.status_code == 200:
        content = response.content

        if not os.path.exists('data'): os.makedirs('data')

        with open(f'data/{os.path.basename(url)}', mode='wb') as csvfile:
            csvfile.write(content)

# armazenar o ano corrente para resgatar os dados referentes à última década
ano_corrente = datetime.datetime.now().year

# baixar todos os arquivos para a pasta data
fluxos = ("IMP", "EXP")
for fluxo in fluxos:
    for ano in range(ano_corrente - 10, ano_corrente):
        download_file(cria_url(fluxo, ano))
        time.sleep(1)

# baixar o arquivo com o mapeamento do código para o nome de cada país
download_file("https://balanca.economia.gov.br/balanca/bd/tabelas/PAIS.csv")

# abrir o arquivo com o mapeamento do código para o nome de cada país
df_pais = pd.read_csv('data/PAIS.csv', encoding='iso-8859-1', header=0, sep=";")[["CO_PAIS", "NO_PAIS"]]

# montar e armazenar os dataframes com valores agregados de exportação e importação por país
dfs = {fluxos[0]: [], fluxos[1]: []}
for fluxo in fluxos:
    for ano in range(ano_corrente - 10, ano_corrente):
        df = pd.read_csv(f"data/{fluxo}_{ano}.csv", header = 0, sep=";")[["CO_ANO", "CO_PAIS", "VL_FOB"]]
        dfs[fluxo].append(df)

    dfs[fluxo] = pd.concat(dfs[fluxo])
    dfs[fluxo] = dfs[fluxo].merge(df_pais, left_on="CO_PAIS", right_on="CO_PAIS")[["CO_ANO", "NO_PAIS", "VL_FOB"]]
    dfs[fluxo] = dfs[fluxo].groupby(["CO_ANO", "NO_PAIS"]).agg({'VL_FOB': "sum"}).reset_index()
    dfs[fluxo] = dfs[fluxo].rename(columns={"CO_ANO": "ano", "NO_PAIS": "país", "VL_FOB": "valor_fob_usd"})
    dfs[fluxo] = dfs[fluxo][dfs[fluxo]["país"] != "A Designar"]
    dfs[fluxo].to_csv(f"data/{fluxo}_{ano_corrente-10}_{ano_corrente-1}.csv", index=False)

# apagar os arquivos intermediários para obter os dois arquivos definitivos
for file in os.listdir("data"):
    if any(file == f"{fluxo}_{ano_corrente-10}_{ano_corrente-1}.csv" for fluxo in fluxos):
        continue
    elif file == ".ipynb_checkpoints":
        continue
    else:
        os.remove(f"data/{file}")

<pre>Agora, crie os DataFrames <b>serie_imp</b> e <b>serie_exp</b> a partir dos arquivos .csv disponíveis na pasta <b>data</b>, do diretório corrente. Execute a célula abaixo para descobrir o nome dos arquivos.</pre>

In [5]:
import datetime

ano_corrente = datetime.datetime.now().year
fluxos = ("IMP", "EXP")

for fluxo in fluxos:
    print(f'serie_{fluxo.lower()}: data/{fluxo}_{ano_corrente - 10}_{ano_corrente-1}.csv')

serie_imp: data/IMP_2015_2024.csv
serie_exp: data/EXP_2015_2024.csv


<pre>Os arquivos contêm os valores <a href="https://pt.wikipedia.org/wiki/Free_on_Board">Free on Board</a> em dólares americanos das importações/exportações por país e ano.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_exp, serie_imp = [pd.read_csv(f'data/{fluxo}_{ano_corrente - 10}_{ano_corrente-1}.csv') for fluxo in ("EXP", "IMP")]
```

<br/>

</details>

<pre>Crie as colunas <b>vl_usd_milhoes</b>, com os valores em milhões de dólares, em ambos os DataFrames <b>serie_imp</b> e <b>serie_exp</b>.

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_imp["vl_usd_milhoes"] = serie_imp["valor_fob_usd"] / 1_000_000
serie_exp["vl_usd_milhoes"] = serie_exp["valor_fob_usd"] / 1_000_000
```

<br/>

</details>

<pre>Ademais, remova as colunas de valor originais (<b>valor_fob_usd</b>), em ambos os DataFrames <b>serie_imp</b> e <b>serie_exp</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_imp = serie_imp.drop(columns='valor_fob_usd')
serie_exp = serie_exp.drop(columns='valor_fob_usd')
```

<br/>

</details>

### Junção, Mescla e Concatenação 🥇

#### Mescla

##### Outer Join

<pre>Agora, junte os DataFrames <b>serie_imp</b> e <b>serie_exp</b> em um único DataFrame <b>serie_exp_imp</b>, contendo as colunas <b>ano</b>, <b>país</b>, <b>vl_usd_milhoes_exp</b> (com os valores da coluna <b>vl_usd_milhoes</b> do DataFrame <b>serie_exp</b>), e <b>vl_usd_milhoes_imp</b> (com os valores da coluna <b>vl_usd_milhoes</b> do DataFrame <b>serie_imp</b>).</pre>

🚨 alguns países podem estar presentes na série de exportações e não na série de importações, e vice-versa. 🚨

<pre>Isto é, queremos que o DataFrame final contenha todos os parceiros comerciais, inclusive os para os quais o Brasil apenas exportou ou dos quais apenas importou.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_exp_imp = serie_exp.merge(serie_imp, on = ["ano", "país"], suffixes = ["_exp", "_imp"], how = "outer")
```

<br/>

</details>

<pre>Em seguida, crie a coluna <b>saldo_usd_milhoes</b>, com a diferença entre as exportações e as importações por país e ano.</pre>

🚨 perceba que, após a junção dos DataFrames, as combinações de chaves que estavam presente em um DataFrame e ausentes em outro podem gerar linhas com `NaN` na coluna `saldo_usd_milhoes`. 🚨

<pre>Isto é, queremos que o valor <b>NaN</b>, tanto da coluna <b>vl_usd_milhoes_exp</b> quanto da coluna <b>vl_usd_milhoes_imp</b>, seja interpretado como <b>0</b> no momento da subtração das exportações pelas importações, para compor o saldo.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
serie_exp_imp['saldo_usd_milhoes'] = serie_exp_imp['vl_usd_milhoes_exp'].fillna(0) - serie_exp_imp['vl_usd_milhoes_imp'].fillna(0)
```

<br/>

</details>

<pre>Por fim, confira quais foram os 10 parceiros comerciais com os quais o Brasil teve o maior saldo no último ano.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
top10_saldos_ultimo_ano = serie_exp_imp[serie_exp_imp["ano"] == serie_exp_imp["ano"].max()]
top10_saldos_ultimo_ano = top10_saldos_ultimo_ano.sort_values(by = "saldo_usd_milhoes", ascending = False)
top10_saldos_ultimo_ano = top10_saldos_ultimo_ano.head(10)
```

<br/>

</details>